# Bicycle JPEG A/B rehearsal and formal source split
Development-only scoring-head rehearsal using the private timing prototype's feature cache. The formal 3,400/800/2,000 manifest is generated from train2017 annotations only; no pixels or scores are read from that split. No FYP commitment decision is made here.

In [ ]:
import csv, hashlib, json, random
from collections import Counter
from pathlib import Path
from time import perf_counter
from copy import deepcopy
import numpy as np
import torch
from torch import nn
from torchvision.ops import box_convert, box_iou
from threadpoolctl import threadpool_limits

start = perf_counter()
torch.set_num_threads(2)
rng = random.Random(5182026)
out = Path("/kaggle/working")
mount = Path("/kaggle/input")
def locate(name):
    for item in mount.glob(f"**/{name}"):
        return item
    raise FileNotFoundError(name)

cache_root = locate("prototype_report.json").parent
proto = json.loads((cache_root/"prototype_report.json").read_text())
assert proto["views"] == 200
dev = {row["image_id"]: row for row in json.loads((cache_root/"development_manifest.json").read_text())}
assert len(dev) == 100
coco = locate("instances_val2017.json").parent
def write_rows(name, rows):
    with (out/name).open("w", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=list(rows[0]))
        writer.writeheader()
        writer.writerows(rows)
with (coco/"instances_train2017.json").open() as f:
    train_data = json.load(f)
cat, = [r["id"] for r in train_data["categories"] if r["name"] == "bicycle"]
crowd = set()
positive = set()
for ann in train_data["annotations"]:
    if ann["category_id"] == cat:
        (crowd if ann["iscrowd"] else positive).add(ann["image_id"])
all_images = {r["id"] for r in train_data["images"]}
del train_data
assert all_images.isdisjoint(dev)
eligible = all_images - crowd
formal = []
for flag, pool in ((1, eligible & positive), (0, eligible - positive)):
    ids = sorted(pool)
    rng.shuffle(ids)
    assert len(ids) >= 3100
    begin = 0
    for split, count in (("train",1700), ("validation",400), ("test",1000)):
        formal.extend({"source_id": i,"bicycle_positive":flag,"split":split} for i in ids[begin:begin+count])
        begin += count
formal.sort(key=lambda r:r["source_id"])
assert len({r["source_id"] for r in formal}) == 6200
write_rows("formal_split.csv", formal)
digest = hashlib.sha256((out/"formal_split.csv").read_bytes()).hexdigest()
print("Formal split frozen:", digest, flush=True)

with (coco/"instances_val2017.json").open() as f:
    sizes = {r["id"]:(r["width"],r["height"]) for r in json.load(f)["images"] if r["id"] in dev}
parts = {"train":[], "validation":[], "test":[]}
dev_rng = random.Random(43109)
for flag in (False, True):
    ids = sorted(i for i,r in dev.items() if bool(r["boxes_xywh"]) == flag)
    dev_rng.shuffle(ids)
    assert len(ids) == 50
    for split, chunk in zip(parts,(ids[:30],ids[30:40],ids[40:])):
        parts[split].extend(chunk)
write_rows("rehearsal_split.csv", [{"source_id":i,"split":s,"bicycle_positive":int(bool(dev[i]["boxes_xywh"]))} for s in parts for i in parts[s]])
print("Development splits:", {s:len(v) for s,v in parts.items()})

In [ ]:
# Each annotated bicycle receives at most one training candidate, chosen greedily by overlap.
# Unselected query duplicates covering a bicycle are ignored in training. Background candidates have IoU < 0.5.
records = {}
gt_n = {}
for i in dev:
    w,h = sizes[i]
    gt = torch.tensor(dev[i]["boxes_xywh"],dtype=torch.float32).reshape(-1,4)
    gt = box_convert(gt / torch.tensor([w,h,w,h]), "xywh","xyxy")
    gt_n[i] = len(gt)
    for q in (90,95):
        r = torch.load(cache_root/"features"/f"{i}_{q}.pt",map_location="cpu",weights_only=True)
        iou = box_iou(box_convert(r["boxes_cxcywh"],"cxcywh","xyxy"),gt)
        labels = torch.zeros(len(iou),dtype=torch.float32)
        candidates = torch.nonzero(iou>=0.5,as_tuple=False)
        used_q,used_gt = set(),set()
        if len(candidates):
            labels[candidates[:,0]] = -1
            for k in torch.argsort(iou[candidates[:,0],candidates[:,1]],descending=True,stable=True).tolist():
                qi,gi = candidates[k].tolist()
                if qi not in used_q and gi not in used_gt:
                    used_q.add(qi)
                    used_gt.add(gi)
                    labels[qi] = 1
        records[i,q] = (r["queries"].float(),r["word_embedding"],r["logits"],iou.numpy(),labels)
        
def metrics(i,q,prob,t):
    overlap = records[i,q][3]
    available = np.ones(gt_n[i],dtype=bool)
    kept = np.flatnonzero(prob >= t)
    order = kept[np.argsort(-prob[kept],kind="stable")]
    for qi in order:
        matches = np.flatnonzero(available & (overlap[qi]>=0.5))
        if len(matches):
            available[matches[np.argmax(overlap[qi,matches])]] = False
    tp = int((~available).sum())
    return tp,len(order)-tp,int(available.sum())

baseline = {k:r[2].sigmoid().numpy() for k,r in records.items()}
grid = np.arange(5,81,5)/100
calibration = []
for t in grid:
    counts = np.sum([metrics(i,q,baseline[i,q],t) for i in parts["validation"] for q in (90,95)],axis=0)
    tp,fp,fn = map(int,counts)
    calibration.append({"threshold":float(t),"validation_f1":2*tp/max(1,2*tp+fp+fn)})
# In an exact F1 tie, retain the lower cutoff.
threshold = max(calibration,key=lambda r:r["validation_f1"])["threshold"]
write_rows("validation_thresholds.csv",calibration)
print("Development threshold",threshold,flush=True)

In [ ]:
def cue_assignment(seed, correlated):
    # Randomise exact frequencies within each bicycle stratum.
    draw = random.Random(seed)
    mapping = {}
    for flag in (False,True):
        ids = sorted(i for i in parts["train"] if bool(gt_n[i]) == flag)
        draw.shuffle(ids)
        n95 = (27 if flag else 3) if correlated else 15
        mapping.update({i:(95 if j<n95 else 90) for j,i in enumerate(ids)})
    assert sum(q==95 for q in mapping.values()) == 30
    return mapping

def train_head(seed, correlated):
    torch.manual_seed(seed)
    head = nn.Sequential(nn.Linear(256,32),nn.ReLU(),nn.Linear(32,256))
    nn.init.zeros_(head[-1].weight)
    nn.init.zeros_(head[-1].bias)
    opt = torch.optim.AdamW(head.parameters(),lr=1e-4)
    assignment = cue_assignment(seed,correlated)
    order_rng = random.Random(seed+771)
    for epoch in range(4):
        ids = sorted(assignment)
        order_rng.shuffle(ids)
        for j in range(0,len(ids),6):
            batch = [records[i,assignment[i]] for i in ids[j:j+6]]
            queries = torch.stack([r[0] for r in batch])
            text = torch.stack([r[1] for r in batch])
            base = torch.stack([r[2] for r in batch])
            labels = torch.stack([r[4] for r in batch])
            logits = base + torch.einsum("bqd,bd->bq",head(queries),text)
            # Preserve the exact frozen logits; residual learns an additive contrastive score.
            bg = nn.functional.softplus(logits[labels==0]).mean()
            fg = logits[labels==1]
            loss = (bg + nn.functional.softplus(-fg).mean())/2 if fg.numel() else bg
            if not torch.isfinite(loss):
                raise ValueError("Non-finite loss")
            opt.zero_grad()
            loss.backward()
            opt.step()
    with torch.no_grad():
        return {(i,q):(r[2]+head(r[0])@r[1]).sigmoid().numpy() for (i,q),r in records.items() if i in parts["test"]}

rows = []
def evaluate(name,seed,pred):
    for condition in ("matched","reversed"):
        for i in parts["test"]:
            matched_q = 95 if gt_n[i] else 90
            q = matched_q if condition=="matched" else 185-matched_q
            tp,fp,fn = metrics(i,q,pred[i,q],threshold)
            rows.append({"model":name,"seed":seed,"condition":condition,"source_id":i,"quality":q,"bicycles":gt_n[i],"tp":tp,"fp":fp,"fn":fn})

evaluate("untouched",0,baseline)
seeds = (11,23,37)
with threadpool_limits(limits=2):
    for seed in seeds:
        for name,correlated in (("A",True),("B",False)):
            evaluate(name,seed,train_head(seed,correlated))
write_rows("rehearsal_per_source.csv",rows)
summary=[]
for model,seed in sorted({(r["model"],r["seed"]) for r in rows}):
    for condition in ("matched","reversed"):
        sub=[r for r in rows if (r["model"],r["seed"],r["condition"])==(model,seed,condition)]
        negatives=[r for r in sub if r["bicycles"]==0]
        total=sum(r["bicycles"] for r in sub)
        missed=sum(r["fn"] for r in sub)
        summary.append({"model":model,"seed":seed,"condition":condition,"bicycles":total,"misses":missed,"miss_rate":missed/total,"false_alarms_per_negative":sum(r["fp"] for r in negatives)/len(negatives)})
write_rows("rehearsal_metrics.csv",summary)

def misses(name,seed,condition):
    return next(r["miss_rate"] for r in summary if (r["model"],r["seed"],r["condition"])==(name,seed,condition))
delta = {seed:100*((misses("A",seed,"reversed")-misses("A",seed,"matched"))-(misses("B",seed,"reversed")-misses("B",seed,"matched"))) for seed in seeds}
report={"formal_sources":len(formal),"formal_split_sha256":digest,"formal_split_seed":5182026,"formal_test_evaluated":False,
        "dev_test_images":len(parts["test"]),"dev_test_bicycles":sum(gt_n[i] for i in parts["test"]),
        "development_threshold":threshold,"delta_pp_by_seed":delta,"mean_delta_pp":float(np.mean(list(delta.values()))),
        "decision_margin_draft_pp":3,"runtime_seconds":perf_counter()-start}
(out/"rehearsal_report.json").write_text(json.dumps(report,indent=2))
print(json.dumps(report,indent=2))